#Fruit Image Classifier — CNN Model

**Objective:** Build a Convolutional Neural Network (CNN) to classify images of **Apple**, **Orange**, and **Banana**.

**Pipeline Overview:**
1. Dataset Exploration & Visualisation
2. Data Cleaning — fix mis-labelled images
3. Class Balancing
4. Image Augmentation
5. CNN Model (Baseline)
6. CNN Model (Improved with Transfer Learning — MobileNetV2)
7. Evaluation & Results
8. Experiment Summary

---
## 0. Imports & Configuration

In [ ]:
import os
import sys
import shutil
import random
import math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter
from PIL import Image
import urllib.request
from urllib.parse import quote
import json
import io

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Sklearn metrics
from sklearn.metrics import classification_report, confusion_matrix

# ── Paths & Configuration ────────────────────────────────────────────────────
GITHUB_OWNER = 'Codemelia'
GITHUB_REPO = 'ml_ca'
GITHUB_BRANCH_CANDIDATES = ['main', 'master']
GITHUB_RAW_BASE = f'https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}'
GITHUB_API_BASE = f'https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents'

# Create local temp directories
LOCAL_TRAIN_DIR = 'temp_train_data'
LOCAL_TEST_DIR = 'temp_test_data'
os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)
os.makedirs(LOCAL_TEST_DIR, exist_ok=True)

CLASSES = ['apple', 'banana', 'orange']
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 30
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Pretty plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 12
})

print('TensorFlow version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))
print('Working with GitHub-sourced images...')


---
## 1. Download Images from GitHub

In [ ]:
def resolve_github_listing(folder_name):
    """
    Resolve the first available branch and return file listing for a GitHub folder.
    """
    for branch in GITHUB_BRANCH_CANDIDATES:
        api_url = f'{GITHUB_API_BASE}/{folder_name}?ref={branch}'
        try:
            with urllib.request.urlopen(api_url, timeout=15) as response:
                items = json.loads(response.read().decode('utf-8'))
            if isinstance(items, list):
                return branch, items
        except Exception:
            continue
    return None, None


def detect_working_branch(folder_name, classes):
    """
    Detect an available branch by probing raw image URLs.
    """
    for branch in GITHUB_BRANCH_CANDIDATES:
        for cls in classes:
            probe_url = f'{GITHUB_RAW_BASE}/{branch}/{folder_name}/{cls}_1.jpg'
            try:
                with urllib.request.urlopen(probe_url, timeout=10) as response:
                    if response.status == 200:
                        return branch
            except Exception:
                continue
    raise RuntimeError(f'Cannot reach raw image URLs for folder {folder_name}')


def download_by_filename_probe(folder_name, target_dir, classes, branch, max_index=500, miss_streak_limit=25):
    counts = {cls: 0 for cls in classes}

    for cls in classes:
        miss_streak = 0
        for i in range(1, max_index + 1):
            filename = f'{cls}_{i}.jpg'
            url = f'{GITHUB_RAW_BASE}/{branch}/{folder_name}/{filename}'
            try:
                with urllib.request.urlopen(url, timeout=10) as response:
                    img_data = response.read()

                save_path = os.path.join(target_dir, cls, filename)
                with open(save_path, 'wb') as f:
                    f.write(img_data)
                counts[cls] += 1
                miss_streak = 0
            except Exception:
                miss_streak += 1
                if miss_streak >= miss_streak_limit and counts[cls] > 0:
                    break

    return counts


def download_images_from_github(folder_name, target_dir, classes):
    """
    Download images from a flat GitHub folder (train/ or test/) and
    classify by filename prefix, e.g. apple_1.jpg -> class apple.
    """
    print(f'Downloading images from GitHub folder: {folder_name}')

    shutil.rmtree(target_dir, ignore_errors=True)
    for cls in classes:
        os.makedirs(os.path.join(target_dir, cls), exist_ok=True)

    branch, items = resolve_github_listing(folder_name)

    if branch and items:
        counts = {cls: 0 for cls in classes}
        for item in items:
            if item.get('type') != 'file':
                continue

            filename = item.get('name', '')
            if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            class_name = filename.split('_', 1)[0].lower()
            if class_name not in classes:
                continue

            download_url = item.get('download_url')
            if not download_url:
                download_url = f"{GITHUB_RAW_BASE}/{branch}/{folder_name}/{quote(filename)}"

            try:
                with urllib.request.urlopen(download_url, timeout=15) as response:
                    img_data = response.read()

                save_path = os.path.join(target_dir, class_name, filename)
                with open(save_path, 'wb') as f:
                    f.write(img_data)
                counts[class_name] += 1
            except Exception as e:
                print(f'Failed to download {filename}: {e}')
    else:
        print('GitHub API listing unavailable, falling back to raw filename probing...')
        branch = detect_working_branch(folder_name, classes)
        counts = download_by_filename_probe(folder_name, target_dir, classes, branch)

    print(f'✓ Using branch: {branch}')
    print(f'✓ Downloaded from {target_dir}:')
    for cls, count in counts.items():
        print(f'   {cls:10s}: {count} images')
    print(f'   {"TOTAL":10s}: {sum(counts.values())} images')

    return counts


# Download training and test images
train_counts = download_images_from_github('train', LOCAL_TRAIN_DIR, CLASSES)
test_counts = download_images_from_github('test', LOCAL_TEST_DIR, CLASSES)


---
## 2. Dataset Exploration

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ('red', 'orange', 'yellow')

for ax, (counts, title) in zip(axes, [(train_counts, 'Training Set'), (test_counts, 'Test Set')]):
    classes_list = list(counts.keys())
    values = list(counts.values())
    bars = ax.bar(classes_list, values, color=colors[:len(classes_list)], edgecolor='black', width=0.5)
    ax.set_title(f'Class Distribution — {title}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Number of Images')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', va='bottom', fontweight='bold')
    ax.set_ylim(0, max(values) * 1.15)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')

---
## 3. Load and Prepare Data for Training

In [ ]:
def load_images_from_directory(directory, classes, img_size=(128, 128)):
    """
    Load all images from class folders with stable class-index mapping.
    Returns: (images_array, labels_array, label_to_class_map)
    """
    images = []
    labels = []
    class_to_label = {cls: idx for idx, cls in enumerate(classes)}
    label_to_class = {idx: cls for cls, idx in class_to_label.items()}

    for class_name in classes:
        class_path = os.path.join(directory, class_name)
        if not os.path.isdir(class_path):
            continue

        for img_file in sorted(os.listdir(class_path)):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(class_path, img_file)
                try:
                    img = Image.open(img_path).convert('RGB')
                    img = img.resize(img_size)
                    img_array = np.array(img, dtype=np.float32) / 255.0

                    images.append(img_array)
                    labels.append(class_to_label[class_name])
                except Exception as e:
                    print(f'Error loading {img_path}: {e}')

    return np.array(images), np.array(labels), label_to_class


# Load training and test data
X_train, y_train, label_map = load_images_from_directory(LOCAL_TRAIN_DIR, CLASSES)
X_test, y_test, _ = load_images_from_directory(LOCAL_TEST_DIR, CLASSES)

print(f'Training data shape: {X_train.shape}')
print(f'Training labels shape: {y_train.shape}')
print(f'Test data shape: {X_test.shape}')
print(f'Test labels shape: {y_test.shape}')
print(f'Label map: {label_map}')

# Convert labels to one-hot encoding
num_classes = len(label_map)
y_train_one_hot = keras.utils.to_categorical(y_train, num_classes)
y_test_one_hot = keras.utils.to_categorical(y_test, num_classes)


---
## 4. Split Training Data into Train/Validation

In [ ]:
from sklearn.model_selection import train_test_split

# Split training data: 80% train, 20% validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train_one_hot, test_size=0.2, random_state=SEED, stratify=y_train
)

print(f'Final Training set: {X_train_split.shape}')
print(f'Validation set: {X_val.shape}')
print(f'Test set: {X_test.shape}')

---
## 5. Model 1: Baseline CNN (WITHOUT Augmentation)

In [ ]:
def build_baseline_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


training_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

# Train baseline model WITHOUT augmentation
print('Training Baseline CNN (NO Augmentation)...')
baseline_model = build_baseline_cnn(X_train_split.shape[1:], num_classes)
baseline_model.summary()

baseline_history = baseline_model.fit(
    X_train_split, y_train_split,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=training_callbacks,
    verbose=1
)

# Evaluate on validation and test sets
baseline_val_loss, baseline_val_acc = baseline_model.evaluate(X_val, y_val, verbose=0)
baseline_test_loss, baseline_test_acc = baseline_model.evaluate(X_test, y_test_one_hot, verbose=0)
print('')
print(f'Baseline Model - Validation Accuracy: {baseline_val_acc:.4f}')
print(f'Baseline Model - Test Accuracy (NO Augmentation): {baseline_test_acc:.4f}')
print(f'Baseline |Val-Test| Gap: {abs(baseline_val_acc - baseline_test_acc):.4f}')


---
## 6. Model 2: CNN WITH Data Augmentation

In [ ]:
# Augmentation for training only
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Train model WITH augmentation
print('Training CNN WITH Data Augmentation...')
augmented_model = build_baseline_cnn(X_train_split.shape[1:], num_classes)

augmented_history = augmented_model.fit(
    train_datagen.flow(X_train_split, y_train_split, batch_size=BATCH_SIZE, shuffle=True, seed=SEED),
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    callbacks=training_callbacks,
    verbose=1
)

# Evaluate on validation and test set (WITHOUT augmentation during evaluation)
augmented_val_loss, augmented_val_acc = augmented_model.evaluate(X_val, y_val, verbose=0)
augmented_test_loss, augmented_test_acc = augmented_model.evaluate(X_test, y_test_one_hot, verbose=0)

print('')
print(f'Augmented Model - Validation Accuracy: {augmented_val_acc:.4f}')
print(f'Augmented Model - Test Accuracy (WITH Augmentation in training): {augmented_test_acc:.4f}')
print(f'Augmented |Val-Test| Gap: {abs(augmented_val_acc - augmented_test_acc):.4f}')


---
## 7. Model 3: MobileNetV2 (Transfer Learning)


In [ ]:
print('Training MobileNetV2 Transfer Learning model...')

MOBILENET_IMG_SIZE = (224, 224)
X_train_mobile = tf.image.resize(X_train_split, MOBILENET_IMG_SIZE).numpy()
X_val_mobile = tf.image.resize(X_val, MOBILENET_IMG_SIZE).numpy()
X_test_mobile = tf.image.resize(X_test, MOBILENET_IMG_SIZE).numpy()

mobilenet_base = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
mobilenet_base.trainable = False

mobilenet_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
], name='mobilenet_augmentation')

mobilenet_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    mobilenet_augmentation,
    layers.Lambda(preprocess_input),
    mobilenet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mobilenet_model.summary()

mobilenet_history = mobilenet_model.fit(
    X_train_mobile, y_train_split,
    validation_data=(X_val_mobile, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=training_callbacks,
    verbose=1
)

mobilenet_val_loss, mobilenet_val_acc = mobilenet_model.evaluate(X_val_mobile, y_val, verbose=0)
mobilenet_test_loss, mobilenet_test_acc = mobilenet_model.evaluate(X_test_mobile, y_test_one_hot, verbose=0)

print('')
print(f'MobileNetV2 - Validation Accuracy: {mobilenet_val_acc:.4f}')
print(f'MobileNetV2 - Test Accuracy: {mobilenet_test_acc:.4f}')
print(f'MobileNetV2 |Val-Test| Gap: {abs(mobilenet_val_acc - mobilenet_test_acc):.4f}')


---
## 8. Compare Results


In [ ]:
# Plot training history for baseline and augmented CNN
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Baseline - Accuracy
axes[0, 0].plot(baseline_history.history['accuracy'], label='Train', linewidth=2)
axes[0, 0].plot(baseline_history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0, 0].axhline(y=baseline_test_acc, color='r', linestyle='--', label=f'Test: {baseline_test_acc:.4f}')
axes[0, 0].set_title('Baseline CNN (NO Augmentation) - Accuracy', fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Baseline - Loss
axes[0, 1].plot(baseline_history.history['loss'], label='Train', linewidth=2)
axes[0, 1].plot(baseline_history.history['val_loss'], label='Validation', linewidth=2)
axes[0, 1].set_title('Baseline CNN (NO Augmentation) - Loss', fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Augmented - Accuracy
axes[1, 0].plot(augmented_history.history['accuracy'], label='Train', linewidth=2)
axes[1, 0].plot(augmented_history.history['val_accuracy'], label='Validation', linewidth=2)
axes[1, 0].axhline(y=augmented_test_acc, color='r', linestyle='--', label=f'Test: {augmented_test_acc:.4f}')
axes[1, 0].set_title('CNN WITH Data Augmentation - Accuracy', fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Augmented - Loss
axes[1, 1].plot(augmented_history.history['loss'], label='Train', linewidth=2)
axes[1, 1].plot(augmented_history.history['val_loss'], label='Validation', linewidth=2)
axes[1, 1].set_title('CNN WITH Data Augmentation - Loss', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Test accuracy comparison across all models
model_names = ['Baseline CNN', 'Augmented CNN', 'MobileNetV2']
test_scores = [baseline_test_acc, augmented_test_acc, mobilenet_test_acc]
val_scores = [baseline_val_acc, augmented_val_acc, mobilenet_val_acc]

plt.figure(figsize=(9, 5))
x = np.arange(len(model_names))
width = 0.35
plt.bar(x - width/2, val_scores, width=width, label='Validation Accuracy', color='#6baed6')
plt.bar(x + width/2, test_scores, width=width, label='Test Accuracy', color='#31a354')
plt.xticks(x, model_names)
plt.ylim(0, 1.0)
plt.ylabel('Accuracy')
plt.title('Validation vs Test Accuracy by Model', fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary comparison
print('')
print('='*60)
print('RESULTS SUMMARY')
print('='*60)
print('')
print('Baseline CNN (NO Augmentation):')
print(f'  Validation Accuracy: {baseline_val_acc:.4f}')
print(f'  Test Accuracy: {baseline_test_acc:.4f}')
print(f'  |Val-Test| Gap: {abs(baseline_val_acc - baseline_test_acc):.4f}')

print('')
print('CNN WITH Data Augmentation:')
print(f'  Validation Accuracy: {augmented_val_acc:.4f}')
print(f'  Test Accuracy: {augmented_test_acc:.4f}')
print(f'  |Val-Test| Gap: {abs(augmented_val_acc - augmented_test_acc):.4f}')

print('')
print('MobileNetV2 (Transfer Learning):')
print(f'  Validation Accuracy: {mobilenet_val_acc:.4f}')
print(f'  Test Accuracy: {mobilenet_test_acc:.4f}')
print(f'  |Val-Test| Gap: {abs(mobilenet_val_acc - mobilenet_test_acc):.4f}')

augmentation_delta = augmented_test_acc - baseline_test_acc
print('')
print(f'Augmentation impact on test accuracy (Aug - NoAug): {augmentation_delta:.4f}')
print('='*60)


---
## 9. Model Predictions & Confusion Matrix (MobileNetV2)


In [ ]:
# Get predictions from MobileNetV2 model
y_pred_probs = mobilenet_model.predict(X_test_mobile)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_one_hot, axis=1)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[label_map[i] for i in range(num_classes)],
            yticklabels=[label_map[i] for i in range(num_classes)],
            ax=ax, cbar_kws={'label': 'Count'})
ax.set_title('Confusion Matrix - MobileNetV2', fontweight='bold', fontsize=14)
ax.set_xlabel('Predicted', fontweight='bold')
ax.set_ylabel('Actual', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification Report
print('')
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=[label_map[i] for i in range(num_classes)]))


---
## 10. Clean Up Temporary Files


In [ ]:
# Optional: Clean up temporary directories after analysis
print('Removing temporary data directories...')
shutil.rmtree(LOCAL_TRAIN_DIR, ignore_errors=True)
shutil.rmtree(LOCAL_TEST_DIR, ignore_errors=True)
print('✓ Cleanup complete')